# 02. Preprocessing Encoding

## 1. Chuẩn bị thư viện

In [1]:
import sys
import warnings
import json
from pathlib import Path

# 1. Tự động tìm PROJECT_ROOT chứa folder 'src'
current_path = Path.cwd().resolve()
PROJECT_ROOT = current_path
for parent in [current_path] + list(current_path.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent
        break

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np

# Import các helper từ src/
from src.data.load_data import load_abalone_data
from src.data.split_data import split_features_target, attach_target
from src.features.feature_engineering import (
    prepare_abalone_feature_groups,
    build_encoded_preprocessor,
    build_standard_scaled_preprocessor,
    transform_with_preprocessor
)

warnings.filterwarnings('ignore')

## 2. Load Data và chia tập huấn luyện

In [6]:
# Tải dữ liệu từ folder gốc
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / 'abalone.csv'
df = load_abalone_data(DATA_PATH)

# Xác định các nhóm đặc trưng
feature_groups = prepare_abalone_feature_groups()
target_col = feature_groups['target_col']
categorical_cols = feature_groups['categorical_cols']
numeric_cols = feature_groups['numeric_cols']

# Chia dữ liệu theo tỷ lệ 70/30
X_train, X_test, y_train, y_test = split_features_target(
    df, target_col, test_size=0.3, random_state=42
)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

Kích thước tập Train: (2923, 8)
Kích thước tập Test: (1254, 8)


## 3. Mã hóa và phân loại

- Phiên bản 1: Chỉ mã hóa biến 'Sex'

In [ ]:
encoded_preprocessor = build_encoded_preprocessor(categorical_cols)
X_train_encoded, X_test_encoded = transform_with_preprocessor(encoded_preprocessor, X_train, X_test)

print("Preview dữ liệu chỉ mã hóa:")
display(X_train_encoded.head(3))

Preview dữ liệu chỉ mã hóa:


,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
2830,1.0,0.0,0.0,0.525,0.430,0.135,0.8435,0.4325,0.1800,0.1815
925,0.0,1.0,0.0,0.430,0.325,0.100,0.3645,0.1575,0.0825,0.1050
3845,0.0,0.0,1.0,0.455,0.350,0.105,0.4160,0.1625,0.0970,0.1450


- Phiên bản 2: Mã hóa và chuẩn hóa các biến số (Standard Scale)

In [ ]:
standard_preprocessor = build_standard_scaled_preprocessor(categorical_cols, numeric_cols)
X_train_standard, X_test_standard = transform_with_preprocessor(standard_preprocessor, X_train, X_test)

print("Preview dữ liệu đã chuẩn hóa (Standard Scale):")
display(X_train_standard.head(3))

Preview dữ liệu đã chuẩn hóa (Standard Scale):


,Sex_F,Sex_I,Sex_M,Length,Diameter,Height,WholeWeight,ShuckedWeight,VisceraWeight,ShellWeight
2830,1.0,0.0,0.0,-0.009546,0.206802,-0.120724,0.014578,0.311054,-0.020982,-0.425408
925,0.0,1.0,0.0,-0.803180,-0.854049,-0.944651,-0.959009,-0.919416,-0.913759,-0.969745
3845,0.0,0.0,1.0,-0.594329,-0.601465,-0.826947,-0.854333,-0.897043,-0.780987,-0.685124


## 4. Lưu dữ liệu đã xử lý

In [9]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

# Ghép lại với biến mục tiêu và lưu file
attach_target(X_train_encoded, y_train).to_csv(processed_dir / 'abalone_train_encoded.csv', index=False)
attach_target(X_test_encoded, y_test).to_csv(processed_dir / 'abalone_test_encoded.csv', index=False)
attach_target(X_train_standard, y_train).to_csv(processed_dir / 'abalone_train_standard.csv', index=False)
attach_target(X_test_standard, y_test).to_csv(processed_dir / 'abalone_test_standard.csv', index=False)

print(f"Đã lưu 4 file dữ liệu vào: {processed_dir}")

Đã lưu 4 file dữ liệu vào: D:\GitHub\comic-store2\TNTT_Repo_NhomYenHoa\AbaloneAge\data\processed
